# reader

> parsing lisp text into an AST

### nbdev prologue

In [ ]:
#| default_exp reader

### Atoms and Literals

In [ ]:
#| export
import ast, re
from fastcore.basics import basic_repr, store_attr, first, last
from compact.types import *

help understand literals in Lisp

In [ ]:
ast.literal_eval("123"), ast.literal_eval('"hello world"'), ast.literal_eval("2+3j")

(123, 'hello world', (2+3j))

In [ ]:
#| export
def Atom(s):
    "parse atoms, default this is a symbol"
    if s == "#t": return True
    if s == "#f": return False
    try: return ast.literal_eval(s)
    except Exception: return Symbol(s)


In [ ]:
Atom("123"), Atom('"hello world"'), Atom("2+3j"), Atom("lambda")

(123, 'hello world', (2+3j), compact.types.Symbol(s='lambda'))

In [ ]:
Atom("+") == Atom("+")

True

In [ ]:
Atom("+") == Atom('"+"')

False

### Reader: Lexer and Parser
parse strings into lisp expressions

In [ ]:
#| export
STR = r'"(?:\\.|[^"\\])*"'  # string with spaces etc.
COMMENT = r';[^\n]*'

DOT = r'\.'

ATOM = r'''[^\s()`',;]+'''

UNQ_SPL = ',@'              # unquote splice
PAREN = '[()]'
QUOTE = "[`',]"

# order matters `,@` needs to show up before the single character `,`
TOKEN_RE = "|".join([STR, COMMENT, DOT, UNQ_SPL, PAREN, QUOTE, ATOM])

def lexer(s): return [t for t in re.findall(TOKEN_RE, s) if not t.startswith(';')]

In [ ]:
lexer("( + 1 2 )")

['(', '+', '1', '2', ')']

In [ ]:
lexer("'( + 1 2 )")

["'", '(', '+', '1', '2', ')']

In [ ]:
lexer("`( + 1 2 )")

['`', '(', '+', '1', '2', ')']

In [ ]:
lexer('`(+ ,@xs 2)')

['`', '(', '+', ',@', 'xs', '2', ')']

In [ ]:
lexer("(+ 1 2)")
# expect: ["(", "+", "1", "2", ")"]

['(', '+', '1', '2', ')']

In [ ]:
lexer("'(1 2 3)")
# expect: ["'", "(", "1", "2", "3", ")"]

["'", '(', '1', '2', '3', ')']

In [ ]:
#| export
def parse_list(sl):
    if Symbol(s=".") in sl: raise SyntaxError("malformed dotted list")
    return sl

In [ ]:
#| export
def parser(toks):
    if not toks: raise SyntaxError("malformed list: unexpected EOF")
    

    t = toks.pop(0)
    if t == ')': raise SyntaxError("malformed list: unexpected )")
    if t == '(':
        sl = []
        while toks and toks[0] != ')': sl.append(parser(toks))
        if not toks: raise SyntaxError("malformed list: missing )")
        toks.pop(0)
        return parse_list(sl)
    sugar = {
        "'": "quote",
        "`": "quasiquote",
        ",": "unquote",
        ",@": "unquote-splicing",
    }
    if t in sugar: return [Symbol(sugar[t]), parser(toks)]

    return Atom(t)

In [ ]:
parser(lexer('`(+ ,@xs 2)'))

compact.types.Pair(car=compact.types.Symbol(s='quasiquote'), cdr=compact.types.Pair(car=compact.types.Pair(car=compact.types.Symbol(s='+'), cdr=compact.types.Pair(car=compact.types.Pair(car=compact.types.Symbol(s='unquote-splicing'), cdr=compact.types.Pair(car=compact.types.Symbol(s='xs'), cdr=())), cdr=compact.types.Pair(car=2, cdr=()))), cdr=()))

In [ ]:
#| export
def parse(s): 
    toks = lexer(s)
    r = parser(toks)
    if not toks: return r
    raise SyntaxError("unexpected tokens after expression")

In [ ]:
parse('`(+ ,@xs 2)')

compact.types.Pair(car=compact.types.Symbol(s='quasiquote'), cdr=compact.types.Pair(car=compact.types.Pair(car=compact.types.Symbol(s='+'), cdr=compact.types.Pair(car=compact.types.Pair(car=compact.types.Symbol(s='unquote-splicing'), cdr=compact.types.Pair(car=compact.types.Symbol(s='xs'), cdr=())), cdr=compact.types.Pair(car=2, cdr=()))), cdr=()))

In [ ]:
parse('`(+ ,@xs 2)')

compact.types.Pair(car=compact.types.Symbol(s='quasiquote'), cdr=compact.types.Pair(car=compact.types.Pair(car=compact.types.Symbol(s='+'), cdr=compact.types.Pair(car=compact.types.Pair(car=compact.types.Symbol(s='unquote-splicing'), cdr=compact.types.Pair(car=compact.types.Symbol(s='xs'), cdr=())), cdr=compact.types.Pair(car=2, cdr=()))), cdr=()))

In [ ]:
#| export
def parse_all(s):
    toks = lexer(s)
    exprs = []
    while toks: exprs.append(parser(toks))
    return exprs

In [ ]:
parse_all("""
(define x 10)
(+ x 5)
""")

[compact.types.Pair(car=compact.types.Symbol(s='define'), cdr=compact.types.Pair(car=compact.types.Symbol(s='x'), cdr=compact.types.Pair(car=10, cdr=()))),
 compact.types.Pair(car=compact.types.Symbol(s='+'), cdr=compact.types.Pair(car=compact.types.Symbol(s='x'), cdr=compact.types.Pair(car=5, cdr=())))]

### nbdev postscript

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()